In [ ]:
import numpy as np
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE" # wei,0621: I added this to run locally
import matplotlib.pyplot as plt
from IPython.display import HTML
plt.rcParams['animation.embed_limit'] = 2**128
import torch
import torch.nn as nn
torch.__version__

In [ ]:
import requests
file = requests.get('https://raw.githubusercontent.com/virgilecheminot/Python-ML-Project/refs/heads/master/system_dynamics.py')
if not os.path.exists('system_dynamics.py'):
    with open('system_dynamics.py', 'wb') as f:
        f.write(file.content)
from system_dynamics import *


All the main functions are stored in `system_dynamics.py`. If possible do not edit the `.py` file, or tell me what to change in it. Although, you can add new functions if you think it is necessary.

In [ ]:
mr = 4 # kg mass of the robot
mb = 10 # kg mass of the base
kr = 10000 # N/m stiffness of the robot arm
kb = 3000 # N/m stiffness of the base
hr = 0.05 # damping ratio of the robot arm
cr = 2 * hr * np.sqrt(kr * mr) # Ns/m damping coefficient of the robot arm
hb = 0.1 # damping ratio of the base
cb = 2 * hb * np.sqrt(kb * mb) # Ns/m damping coefficient of the base
M = np.array([[mr, mr], [0, mb]])
C = np.array([[cr, 0], [-cr, cb]])
K = np.array([[kr, 0], [-kr, kb]])
robot = System(M, C, K,['xr','xb'])

In [ ]:
robot.res_freq()

In [ ]:
x0 = np.array([0, 0, 0, 0])  # Initial conditions: [xr, xb, xr_dot, xb_dot]
time = np.linspace(0, 10, 1000)  # Time vector from 0 to 10 seconds
f0 = 0  # Starting frequency of the chirp (Hz)
f1 = 2.5  # Ending frequency of the chirp (Hz)
T = time[-1] - time[0]  # Total duration of the chirp
amplitude = 0.9 * kr  # Amplitude of the force applied to the robot arm

f = lambda t: chirp_hold(t, f0, f1, T, amplitude)[0]
# f = lambda t: np.sin(2 * np.pi * 0.2 * t) * 0.1 * kr
u = lambda t: np.array([f(t), -f(t)])
f_disp = [f(t) for t in time]
freq_disp = [chirp_hold(t, f0, f1, T, amplitude)[1] for t in time]

fig, ax1 = plt.subplots(1, 1, figsize=(8, 4))
ax1.plot(time, f_disp, label='Chirp Force on Robot Arm')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Force (N)')
ax1.set_title('Chirp + Hold Force on Robot Arm')
ax2 = ax1.twinx()
ax2.plot(time, freq_disp, color='red', label='Frequency (Hz)', linestyle='--')
# ax2.plot(time, np.full_like(time, 0.2), color='red', label='Frequency (Hz)', linestyle='--')
ax2.set_ylabel('Frequency (Hz)')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
fig.tight_layout()
plt.show()

In [ ]:
robot.plot_response(x0, time, u)

In [ ]:
outreach, ani = evaluate_performance(robot, 10, 2.5, 8, kr, plot_response=True)
print(f'Performance outreach: {outreach:.2f} cm')
if ani is not None:
    display(HTML(ani.to_jshtml()))

In [ ]:
np.random.seed(42)

config_list = []
for _ in range(500):
    hr = np.random.uniform(0.01, 0.1)
    hb = np.random.uniform(0.05, 0.2)
    # kr = np.abs(np.random.normal(10000, 2000))
    kr = np.random.uniform(7500, 15000)
    # kb = np.abs(np.random.normal(3000, 750))
    kb = np.random.uniform(1000, 6000)
    mr = np.random.uniform(1, 7)
    amplitude = np.random.uniform(5, 10)
    f1 = np.random.uniform(0.1, 5)
    T = np.random.uniform(5, 15)
    config_list.append({'hr': hr, 'hb': hb, 'kr': kr, 'kb': kb, 'mr': mr, 'amplitude': amplitude, 'f1': f1, 'T': T})

X_train, y_train, _ = generate_training_data(config_list, plot_samples=False)

X_train data is the config list normalised by the "central" value of the training set:

<center>
<table>
<tr>
    <td>hr: 0.05</td>
    <td>hb: 0.1</td>
    <td>kr: 10000 N/m</td>
    <td>kb: 3000 N/m</td>
</tr>
<tr>
    <td>mr: 4 kg</td>
    <td>amplitude: 10 cm</td>
    <td>f1: 2.5 Hz</td>
    <td>T: 10 s</td>
</tr>
</table>
</center>

Bellow is just a tentative but I really am not sure of what I am doing with `nn.Module`. Expecially with the dimensions of the tensors. I don't know what activation function to use, how many layers, how many neurons per layer, etc...

In [ ]:
class RobotNN(nn.Module):
    """Simple MLP to predict outreach from system parameters."""

    def __init__(self, input_size=8, hidden_sizes=(64, 32, 16), output_size=1):
        super().__init__()
        layers = []
        last_size = input_size

        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(last_size, hidden_size))
            layers.append(nn.ReLU())
            last_size = hidden_size

        layers.append(nn.Dropout(p=0.1))
        layers.append(nn.Linear(last_size, output_size))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(-1)

In [ ]:
dataset = torch.utils.data.TensorDataset(X_train, y_train)
train_loader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True)

model = RobotNN()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 200
losses = [] # This list will now store the average loss per epoch
for epoch in range(epochs):
    model.train()
    current_epoch_batch_losses = [] # Temporary list to store losses for batches within the current epoch
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch)
        # Squeeze y_batch to match predictions shape for MSELoss
        loss = criterion(predictions, y_batch.squeeze(-1))
        current_epoch_batch_losses.append(loss.item()) # Append batch loss to temporary list
        loss.backward()
        optimizer.step()
    avg_loss_epoch = sum(current_epoch_batch_losses) / len(current_epoch_batch_losses) # Calculate average loss for the epoch
    losses.append(avg_loss_epoch) # Append the average epoch loss to the main losses list
    # print(f'Epoch {epoch+1}/{epochs}, Loss: {avg_loss_epoch:.4f}')

torch.save(model.state_dict(), 'robot_nn_model.pth')

plt.plot(losses)
plt.yscale('log')

In [ ]:
def predict_outreach(model, cfg):
    model.load_state_dict(torch.load('robot_nn_model.pth'))
    model.eval()
    x_tensor = torch.FloatTensor([[
        cfg['hr']/0.05,
        cfg['hb']/0.1,
        cfg['kr']/10000.0,
        cfg['kb']/3000.0, # Wei,0621: I changed 2000 to 3000
        cfg['mr']/4.0,
        cfg['amplitude']/10.0,
        cfg['f1']/2.5,
        cfg['T']/10.0
    ]])
    with torch.no_grad():
        predicted_outreach = model(x_tensor).item()
    return predicted_outreach

In [ ]:
test_cfg = {
    'hr': 0.05,
    'hb': 0.1,
    'kr': 12000,
    'kb': 4000,
    'mr': 4,
    'amplitude': 9,
    'f1': 4,
    'T': 10
}
M_test = np.array([[test_cfg['mr'], test_cfg['mr']], [0, 10]])
cr_test = 2 * test_cfg['hr'] * np.sqrt(test_cfg['kr'] * test_cfg['mr'])
cb_test = 2 * test_cfg['hb'] * np.sqrt(test_cfg['kb'] * 10)
C_test = np.array([[cr_test, 0], [-cr_test, cb_test]])
K_test = np.array([[test_cfg['kr'], 0], [-test_cfg['kr'], test_cfg['kb']]])

robotTest = System(M_test, C_test, K_test,['xr','xb'])

outreach, _ = evaluate_performance(robotTest, 10, 2.5, 8, kr, plot_response=False)
print(f'Real outreach: {outreach:.2f} cm')
outreachNN = predict_outreach(model, test_cfg)
print(f'Predicted outreach: {outreachNN:.2f} cm')